# IAC Agentic AI — Task 2: Master Dataset & ML Dataset Engineering
**Project:** Cloud Counselage Foundation — Agentic AI Outreach & Enrollment System  
**Input:** `clean_census.csv`, `clean_gsdp.csv`, `clean_unemployment.csv`  
**Output:** `master_dataset.csv`, `ml_dataset.csv`, `scaled_dataset.csv`  

This notebook covers Tasks 1–7 as specified:
- Column audit & justification
- Column removal
- Data quality verification
- Inner-join merge
- Master dataset creation
- ML feature selection & justification
- StandardScaler normalization

In [ ]:
# ── 0. IMPORTS ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import sqlite3
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

---
## TASK 1 — Column Audit: What to Keep, What to Remove, and Why

### Census Dataset (12 columns)

| Column | Decision | Justification |
|---|---|---|
| `state_name` | ✅ KEEP | Primary join key across all three datasets |
| `area_sq_km` | ✅ KEEP | Geographic scale; needed for density cross-validation and dashboard mapping |
| `total_population` | ✅ KEEP | Core demographic size metric; drives outreach prioritization |
| `national_share_pct` | ❌ **REMOVE** | Mathematically derived from `total_population` — provides zero new information. Including it in K-Means introduces multicollinearity and distorts cluster geometry. |
| `decadal_growth_rate` | ✅ KEEP | Growth trajectory distinguishes high-growth vs stagnant states; useful for segment storytelling even if not used in clustering |
| `rural_population` | ✅ KEEP (master only) | Absolute count for dashboard reporting; excluded from ML because it is the raw numerator behind `rural_pct` — redundant for clustering |
| `rural_pct` | ✅ KEEP | Normalized rural share; valid clustering feature — channel strategy (SMS/community vs digital) depends heavily on this |
| `urban_population` | ✅ KEEP (master only) | Absolute count for reporting; excluded from ML for same reason as `rural_population` |
| `urban_pct` | ✅ KEEP | Normalized urban share; complementary to `rural_pct` — together they capture the rural/urban split fully |
| `pop_density_per_sq_km` | ✅ KEEP | Density proxy for outreach logistics — dense urban clusters vs sparse rural blocks need different campaign delivery |
| `sex_ratio` | ✅ KEEP (master only) | Useful for reporting and demographic analysis; excluded from ML — sex ratio does not meaningfully differentiate youth outreach priority clusters |
| `literacy_rate_pct` | ✅ KEEP | Strongest single indicator of IAC program readiness; high-literacy states are ready for digital onboarding, low-literacy need in-person/WhatsApp/poster campaigns |

---
### GSDP Dataset (5 columns)

| Column | Decision | Justification |
|---|---|---|
| `state_name` | ✅ KEEP | Join key |
| `gsdp_current_cr_2022_23` | ✅ KEEP | Current-price GSDP = economic size in nominal terms; most interpretable to non-technical stakeholders and to the Text-to-SQL agent's natural language answers |
| `gsdp_growth_current_pct_2022_23` | ✅ KEEP | Economic momentum; separates structurally lagging states from fast-growing ones |
| `gsdp_constant_cr_2022_23` | ❌ **REMOVE** | Inflation-adjusted GSDP is valuable for macroeconomic research but adds multicollinearity for clustering — it is highly correlated with current-price GSDP. One price-base is sufficient. |
| `gsdp_growth_constant_pct_2022_23` | ❌ **REMOVE** | Same reasoning — the real growth rate and nominal growth rate move together for our 15-state sample. Keeping both would double-weight economic growth in the cluster distance calculation. |

---
### Unemployment Dataset (6 columns)

| Column | Decision | Justification |
|---|---|---|
| `state_name` | ✅ KEEP | Join key |
| `unemp_rate_2019_20` | ❌ **REMOVE** | Pre-COVID baseline — structurally different from post-COVID labour market. Including it anchors clusters to a labour market that no longer exists. |
| `unemp_rate_2020_21` | ❌ **REMOVE** | Peak COVID disruption year — an anomaly, not a structural signal. |
| `unemp_rate_2021_22` | ❌ **REMOVE** | Still transitional/recovery year. |
| `unemp_rate_2022_23` | ❌ **REMOVE** | Near-current but superseded by 2023-24 data. Keeping multiple years of unemployment rates highly correlated with each other degrades K-Means cluster quality through redundancy. |
| `unemp_rate_2023_24` | ✅ KEEP | Most recent, most representative of current labour market reality. This is the signal the AI agent should use to identify underserved youth. |

In [ ]:
# ── TASK 2: LOAD AND REMOVE COLUMNS ──────────────────────────────────────────

# --- Census ---
census = pd.read_csv('clean_census.csv')
print(f'Census loaded: {census.shape}')

# Remove national_share_pct — derived from total_population, adds multicollinearity
census.drop(columns=['national_share_pct'], inplace=True)
print(f'Census after column removal: {census.shape}')
print('Remaining columns:', census.columns.tolist())

In [ ]:
# --- GSDP ---
gsdp = pd.read_csv('clean_gsdp.csv')
print(f'GSDP loaded: {gsdp.shape}')

# Remove constant-price columns — correlated with current-price, redundant for clustering
gsdp.drop(columns=['gsdp_constant_cr_2022_23', 'gsdp_growth_constant_pct_2022_23'], inplace=True)
print(f'GSDP after column removal: {gsdp.shape}')
print('Remaining columns:', gsdp.columns.tolist())

In [ ]:
# --- Unemployment ---
unemp = pd.read_csv('clean_unemployment.csv')
print(f'Unemployment loaded: {unemp.shape}')

# Remove historical years — keep only 2023-24 (most current signal)
unemp.drop(
    columns=['unemp_rate_2019_20', 'unemp_rate_2020_21',
             'unemp_rate_2021_22', 'unemp_rate_2022_23'],
    inplace=True
)
print(f'Unemployment after column removal: {unemp.shape}')
print('Remaining columns:', unemp.columns.tolist())

---
## TASK 3 — Data Quality Verification

In [ ]:
# ── TASK 3: QUALITY CHECK FUNCTION ────────────────────────────────────────────
def quality_report(df, name):
    print(f'\n{"="*60}')
    print(f'QUALITY REPORT: {name}')
    print(f'{"="*60}')
    print(f'Shape                  : {df.shape}')
    print(f'Duplicate rows         : {df.duplicated().sum()}')
    print(f'Duplicate state_names  : {df["state_name"].duplicated().sum()}')
    print(f'\nMissing values per column:')
    print(df.isnull().sum().to_string())
    print(f'\nData types:')
    print(df.dtypes.to_string())

    # Check for leftover symbols in object columns
    for col in df.select_dtypes(include='object').columns:
        for sym in [',', '%', '₹', '  ']:
            hits = df[col].astype(str).str.contains(sym, regex=False).sum()
            if hits > 0:
                print(f'  ⚠ Symbol "{sym}" found {hits}x in column "{col}"')

    # Leading/trailing whitespace in state names
    ws_issues = (df['state_name'] != df['state_name'].str.strip()).sum()
    if ws_issues:
        print(f'  ⚠ Whitespace issues in state_name: {ws_issues}')
    else:
        print('  ✓ state_name: no whitespace issues')

    print(f'\nState names:')
    print(sorted(df['state_name'].tolist()))

quality_report(census, 'CENSUS')
quality_report(gsdp,   'GSDP')
quality_report(unemp,  'UNEMPLOYMENT')

In [ ]:
# ── TASK 3: APPLY FIXES ───────────────────────────────────────────────────────
# These datasets were already cleaned in Stage 1, but we re-apply defensively
# so this notebook is independently runnable.

def standardize_state_name(series):
    """Strip whitespace, fix known historical name variants."""
    rename_map = {
        'Orissa'     : 'Odisha',
        'Uttaranchal': 'Uttarakhand',
    }
    return series.str.strip().replace(rename_map)

def clean_numeric_col(series):
    """Remove commas, %, ₹, coerce to float."""
    return (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('%', '', regex=False)
        .str.replace('₹', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
        .pipe(pd.to_numeric, errors='coerce')
    )

for df in [census, gsdp, unemp]:
    df['state_name'] = standardize_state_name(df['state_name'])
    for col in df.select_dtypes(include=['object']).columns:
        if col != 'state_name':
            df[col] = clean_numeric_col(df[col])

# Ensure all numeric columns are float64
for df in [census, gsdp, unemp]:
    for col in df.columns:
        if col != 'state_name':
            df[col] = pd.to_numeric(df[col], errors='coerce')

print('Standardization applied to all three datasets ✓')

In [ ]:
# ── TASK 3: POST-FIX QUALITY CONFIRMATION ─────────────────────────────────────
print('Post-fix missing values:')
for df, name in [(census,'CENSUS'),(gsdp,'GSDP'),(unemp,'UNEMPLOYMENT')]:
    total_missing = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    state_dupes = df['state_name'].duplicated().sum()
    status = '✓' if total_missing == 0 and dupes == 0 and state_dupes == 0 else '⚠'
    print(f'  {status} {name}: missing={total_missing}, dup_rows={dupes}, dup_states={state_dupes}')

---
## TASK 4 — Merge Datasets (Inner Join on `state_name`)

In [ ]:
# ── TASK 4: INNER JOIN ────────────────────────────────────────────────────────
# Inner join ensures only states that exist in ALL three datasets are kept.
# After filtering each to the same 15 states upstream, this should always
# produce exactly 15 rows.

master = (
    census
    .merge(gsdp,  on='state_name', how='inner')
    .merge(unemp, on='state_name', how='inner')
)

print(f'Merged shape: {master.shape}')
print(f'States in merged dataset: {len(master)}')

# ── VERIFICATION ──────────────────────────────────────────────────────────────
TARGET_STATES = [
    'Uttar Pradesh', 'Maharashtra', 'Bihar', 'West Bengal',
    'Madhya Pradesh', 'Tamil Nadu', 'Rajasthan', 'Karnataka',
    'Gujarat', 'Andhra Pradesh', 'Odisha', 'Telangana',
    'Kerala', 'Jharkhand', 'Assam'
]

assert len(master) == 15, f'Expected 15 states, got {len(master)}'
assert master.duplicated().sum() == 0, 'Duplicate rows found in merged dataset'
assert master.isnull().sum().sum() == 0, 'Missing values found in merged dataset'

missing_states = set(TARGET_STATES) - set(master['state_name'])
assert not missing_states, f'These target states are missing: {missing_states}'

print()
print('✓ Exactly 15 states present')
print('✓ Zero duplicate rows')
print('✓ Zero missing values')
print('✓ All 15 target states confirmed')

---
## TASK 5 — Create Master Dataset

In [ ]:
# ── TASK 5: SELECT EXACT COLUMNS FOR MASTER DATASET ──────────────────────────
MASTER_COLS = [
    'state_name',
    'area_sq_km',
    'total_population',
    'decadal_growth_rate',
    'rural_population',
    'rural_pct',
    'urban_population',
    'urban_pct',
    'pop_density_per_sq_km',
    'sex_ratio',
    'literacy_rate_pct',
    'gsdp_current_cr_2022_23',
    'gsdp_growth_current_pct_2022_23',
    'unemp_rate_2023_24'
]

master_dataset = master[MASTER_COLS].copy()

print('=== MASTER DATASET ===')
print(f'Shape: {master_dataset.shape}')
print(f'Columns ({len(master_dataset.columns)}):')
for c in master_dataset.columns:
    print(f'  {c} — {master_dataset[c].dtype}')
print(f'\nMissing values: {master_dataset.isnull().sum().sum()}')
print(f'Duplicate rows: {master_dataset.duplicated().sum()}')

In [ ]:
# ── TASK 5: DISPLAY MASTER DATASET ───────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)
print('First 5 rows:')
master_dataset.head()

In [ ]:
# ── TASK 5: SAVE MASTER DATASET ───────────────────────────────────────────────
master_dataset.to_csv('master_dataset.csv', index=False)
print('Saved → master_dataset.csv ✓')

# ── TASK 5: LOAD INTO SQLITE ──────────────────────────────────────────────────
# This is the table the Text-to-SQL agent will query.
conn = sqlite3.connect('iac_outreach.db')
master_dataset.to_sql('state_demographics', conn, if_exists='replace', index=False)
conn.commit()

# Verify with a sample Text-to-SQL-style natural language query
result = pd.read_sql("""
    SELECT state_name, literacy_rate_pct, gsdp_current_cr_2022_23, unemp_rate_2023_24
    FROM state_demographics
    WHERE literacy_rate_pct < 70
    ORDER BY unemp_rate_2023_24 DESC
""", conn)
print()
print('SQLite verification — states with literacy < 70% ordered by unemployment:')
print(result.to_string(index=False))
conn.close()
print('\nSQLite table state_demographics updated ✓')

---
## TASK 6 — Create ML Dataset (K-Means Feature Selection)

### Feature Selection Decisions

**INCLUDED in K-Means (8 features):**

| Feature | Why included |
|---|---|
| `total_population` | Scale of the youth population addressable by outreach — directly determines campaign reach and resource allocation per cluster |
| `literacy_rate_pct` | Strongest discriminator of IAC program readiness and outreach channel suitability |
| `rural_pct` | Determines outreach mode — high rural % → community posters, in-person, SMS; low → digital-first |
| `urban_pct` | Complement to rural_pct; together they fully describe the settlement split |
| `pop_density_per_sq_km` | Affects campaign logistics — dense areas suit WhatsApp/SMS blast; sparse need community champions |
| `gsdp_current_cr_2022_23` | Economic capacity of state — wealthier states have different digital infrastructure and youth aspirations |
| `gsdp_growth_current_pct_2022_23` | Economic momentum — growing states produce different opportunity signals than stagnating ones |
| `unemp_rate_2023_24` | Youth opportunity gap — high unemployment is the core signal for IAC intervention need |

---

**EXCLUDED from K-Means (with justification):**

| Feature | Why excluded |
|---|---|
| `state_name` | Categorical string — K-Means operates on Euclidean distance in continuous numeric space; text has no distance meaning |
| `area_sq_km` | Geographic size alone does not characterize a youth outreach cluster. A large, sparse, poor state (Rajasthan) and a large, dense, rich state (Maharashtra) should NOT cluster together just because they share large area. |
| `rural_population` | Raw count — redundant given `rural_pct`. Including both means the same information is weighted twice in distance calculation, biasing cluster geometry. |
| `urban_population` | Same reason as `rural_population` — redundant with `urban_pct`. |
| `sex_ratio` | Does not meaningfully differentiate youth outreach priority segments at the state level. Sex ratio variation across our 15 states is relatively narrow (879–1084) and does not align with any actionable outreach dimension. |
| `decadal_growth_rate` | 2001-2011 demographic growth data — a decade+ old and structurally different from current conditions. Does not reflect the current youth population distribution that the AI agent is targeting. |

In [ ]:
# ── TASK 6: CREATE ML DATASET ────────────────────────────────────────────────
ML_FEATURES = [
    'total_population',
    'literacy_rate_pct',
    'rural_pct',
    'urban_pct',
    'pop_density_per_sq_km',
    'gsdp_current_cr_2022_23',
    'gsdp_growth_current_pct_2022_23',
    'unemp_rate_2023_24'
]

# ml_dataset: numeric features only, no state_name
# We keep state_name in a separate index series for labeling cluster outputs later
state_labels = master_dataset['state_name'].reset_index(drop=True)
ml_dataset   = master_dataset[ML_FEATURES].copy().reset_index(drop=True)

print('=== ML DATASET ===')
print(f'Shape: {ml_dataset.shape}')
print(f'Columns: {ml_dataset.columns.tolist()}')
print(f'Missing values: {ml_dataset.isnull().sum().sum()}')
print()
print('Descriptive statistics:')
ml_dataset.describe().round(2)

In [ ]:
# ── TASK 6: CHECK FEATURE VARIANCE ───────────────────────────────────────────
# Features with near-zero variance contribute nothing to clustering.
# All features should show meaningful spread.
print('Feature standard deviations (pre-scaling):')
for col in ml_dataset.columns:
    std = ml_dataset[col].std()
    flag = '⚠ LOW VARIANCE' if std < 0.01 else '✓'
    print(f'  {flag}  {col:<40} std = {std:.4f}')

In [ ]:
# ── TASK 6: SAVE ML DATASET ───────────────────────────────────────────────────
# We save with state_name as a reference column so it can be used
# to label cluster outputs — but it will NOT be passed to the scaler or K-Means.
ml_with_labels = ml_dataset.copy()
ml_with_labels.insert(0, 'state_name', state_labels)
ml_with_labels.to_csv('ml_dataset.csv', index=False)
print('Saved → ml_dataset.csv ✓')
print(f'Shape: {ml_with_labels.shape}  (state_name included as reference only)')
ml_with_labels.head()

---
## TASK 7 — Normalization with StandardScaler

**Why StandardScaler, not MinMaxScaler?**

- K-Means uses **Euclidean distance**. Without scaling, high-magnitude features like `gsdp_current_cr_2022_23` (values in millions) completely dominate the distance calculation, making `literacy_rate_pct` (values 60–95) effectively invisible to the clustering algorithm.
- **StandardScaler** transforms each feature to **mean=0, std=1**. This gives every feature equal weight in the distance calculation regardless of its original scale.
- MinMaxScaler is sensitive to outliers (Uttar Pradesh's population would compress all other states toward 0). StandardScaler is more robust.
- The original `ml_dataset` values are **NOT overwritten** — `scaled_dataset` is a separate object.

In [ ]:
# ── TASK 7: APPLY STANDARDSCALER ─────────────────────────────────────────────
scaler = StandardScaler()

# Fit and transform ONLY the numeric ml_dataset (no state_name)
scaled_values = scaler.fit_transform(ml_dataset)

# Build scaled_dataset dataframe with state labels for interpretability
scaled_dataset = pd.DataFrame(
    scaled_values,
    columns=ML_FEATURES
)
scaled_dataset.insert(0, 'state_name', state_labels)

print('=== SCALED DATASET ===')
print(f'Shape: {scaled_dataset.shape}')
print(f'Missing values: {scaled_dataset.isnull().sum().sum()}')
print()
print('Post-scaling statistics (each feature should have mean≈0, std≈1):')
stats = scaled_dataset[ML_FEATURES].agg(['mean','std']).round(4)
print(stats.to_string())

In [ ]:
# ── TASK 7: VERIFY ORIGINAL IS UNCHANGED ─────────────────────────────────────
# ml_dataset must still contain original unscaled values
print('Original ml_dataset (first row, first 3 features — should be unscaled):')
print(ml_dataset[['total_population', 'literacy_rate_pct', 'gsdp_current_cr_2022_23']].iloc[0].to_dict())
print()
print('scaled_dataset (first row, first 3 features — should be z-scores near 0):')
print(scaled_dataset[['total_population', 'literacy_rate_pct', 'gsdp_current_cr_2022_23']].iloc[0].to_dict())
print()
print('✓ Original values preserved — scaling did NOT overwrite ml_dataset')

In [ ]:
# ── TASK 7: DISPLAY SCALED DATASET ───────────────────────────────────────────
print('Scaled dataset (all 15 states):')
scaled_dataset.round(4)

In [ ]:
# ── TASK 7: SAVE SCALED DATASET ───────────────────────────────────────────────
scaled_dataset.to_csv('scaled_dataset.csv', index=False)
print('Saved → scaled_dataset.csv ✓')

In [ ]:
# ── PIPELINE SUMMARY ──────────────────────────────────────────────────────────
print('=' * 60)
print('PIPELINE COMPLETE — ALL TASKS DONE')
print('=' * 60)
print()
print('Files generated:')
print(f'  master_dataset.csv     → {master_dataset.shape[0]} rows × {master_dataset.shape[1]} cols  (SQLite/Dashboard/Agents)')
print(f'  ml_dataset.csv         → {ml_with_labels.shape[0]} rows × {ml_with_labels.shape[1]} cols  (K-Means input, unscaled)')
print(f'  scaled_dataset.csv     → {scaled_dataset.shape[0]} rows × {scaled_dataset.shape[1]} cols  (K-Means input, StandardScaler)')
print(f'  iac_outreach.db        → SQLite: table state_demographics updated')
print()
print('Column counts:')
print(f'  master_dataset  : {len(master_dataset.columns)} columns (all features + reporting fields)')
print(f'  ml_dataset      : {len(ML_FEATURES)} numeric features + state_name label')
print(f'  scaled_dataset  : {len(ML_FEATURES)} scaled features + state_name label')
print()
print('Next step: K-Means clustering on scaled_dataset (Day 3)')